In [1]:
import torch
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else "cpu"

In [2]:
# load a PDF file
import requests
import os

url = 'https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf'

# Get PDF docuement path
file_name = 'topics_6.pdf'
output_path = f'{file_name}'

if not os.path.exists(output_path):
    print(f'[INFO] file doesnt exist, downloading...')
    try:
        response = requests.get(url)

        # check if request was successful
        if response.status_code == 200:
            # open the file and save it
            with open(output_path, 'wb') as f:
                f.write(response.content)

            print(f"[INFO] file has been downloaded and saved as {file_name}");
        else:
            print(f'[INFO] Failed to download the file. Status code: {response.status_code}')

    except Exception as e:
        print(f"An error occurred: {e}")

else:
    print(f'File {output_path} exists')



File topics_6.pdf exists


In [3]:
import torch
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm

In [4]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\d1990\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\d1990\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
def preprocess(docs):
    result = []
    stop_words = set(stopwords.words('english'))
    for text in docs:
        #convert to lowercase, then replace non alpha chars with empty
        text = re.sub(r'[^a-z\s]', '', text.lower())
        tokens = word_tokenize(text)
        # remove stopwords
        filtered_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
        result.append(filtered_tokens)
    return result

In [7]:
import fitz #pymupdf
from tqdm.auto import tqdm

def open_and_read_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = preprocess([page.get_text()])[0]
        if (len(text) > 10):
            pages_and_texts.append({
                "text": text,
                "raw_text": page.get_text(),
                "page_number": len(pages_and_texts) + 1
                }) 
        
    return pages_and_texts


pages_and_texts = open_and_read_pdf(output_path)

0it [00:00, ?it/s]

6it [00:00, 34.31it/s]


In [8]:
#build a vocab
vocab = set()

# Loop through num_sentence_chunk_size and split sentences into chunks
for item in tqdm(pages_and_texts):
    topic = item["text"]
    words = topic
    for w in words:
        if (len(w.strip()) > 0):
            vocab.add(w)

100%|██████████| 6/6 [00:00<00:00, 3014.23it/s]


In [10]:
vocab_size = len(vocab)
doc_size = len(pages_and_texts)

#create a mapping from characters to integers
stoi = {s: i for i, s in enumerate(vocab)}
itos = {i : s for s, i in stoi.items()}
encode = lambda s : [stoi[c] for c in s] #encoder: take a sting, output a list of integers
decode = lambda l: ' '.join([itos[i] for i in l]) #decoder: take a list of integers, output a string

In [11]:
def create_bow(docs):
    bow = torch.zeros(vocab_size, doc_size, dtype=torch.float32, device=device)

    for idx , doc in enumerate(docs):
        for word in doc['text']:
            bow[stoi[word.lower()], idx] += 1

    return bow

bow = create_bow(docs=pages_and_texts)

In [12]:
bow

tensor([[0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        ...,
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0., 1.]])

In [13]:
def svd_k(bow, k = 2):
    col_space, S, row_space = torch.linalg.svd(bow, full_matrices=False)
    U = col_space[:, :k]
    S = torch.diag(S[:k])
    V_t = row_space[:k, :]

    return U, S, V_t 

U_k, S_k, V_k = svd_k(bow=bow)

In [14]:
print(U_k.shape, S_k.shape, V_k.shape)

torch.Size([534, 2]) torch.Size([2, 2]) torch.Size([2, 6])


In [15]:
bow.shape, (U_k @ S_k @ V_k).shape

(torch.Size([534, 6]), torch.Size([534, 6]))

In [ ]:
torch.all(bow == (U_k @ S_k @ V_k))

tensor(False)

In [28]:
def query_lsi(query):
    tokens = word_tokenize(query.lower())
    q_vec = torch.zeros(vocab_size, dtype=torch.float32)
    for w in tokens:
        q_vec[stoi[w]] += 1
    
    # this query is now vocab space (column space)
    # we need to project it docs space (row space)
    S_k_inv = torch.inverse(S_k)

    lsi_vec = q_vec.T @ U_k @ S_k_inv

    print(lsi_vec.shape, V_k.shape)


    # we will do cosine similarity
    similarities = F.cosine_similarity(lsi_vec.unsqueeze(0), V_k.T, dim=1)

    indices = torch.argsort(similarities, descending=True)

    return [(pages_and_texts[idx]["raw_text"], similarities[idx].item()) for idx in indices]
    

results = query_lsi("threats stargazing")

for res, sim in results:
    print(f"Similarity: {sim*100:.2f}%")
    print(res[:125].strip())
    print("-----")

torch.Size([2]) torch.Size([2, 6])
Similarity: 99.49%
Cybersecurity is the practice of protecting systems, networks, and data from digital attacks, 
which are often aimed at acces
-----
Similarity: 91.52%
A balanced diet provides the necessary energy and nutrients for optimal physical and mental 
well-being. It is not about ri
-----
Similarity: 88.17%
The Internet evolved from a military project into a global communication backbone through 
several key technological advanc
-----
Similarity: 86.53%
Stargazing is a rewarding hobby that allows you to connect with the wider universe. You can 
start with minimal equipment.
-----
Similarity: 20.93%
Effective study techniques help optimize learning and retention. Two highly effective, evidence-
based methods are active r
-----
Similarity: 19.41%
This topic is a great opportunity for creativity. You can select existing works you love 
or include some of your own writi
-----
